In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.options.display.float_format = '{:,.4f}'.format

In [4]:
# Расчет PSI

def _psi(expected: np.ndarray, actual: np.ndarray, bucket_type: str = "bins", n_bins: int = 10) -> float:
    """Calculate PSI between two distributions."""
    breakpoints = np.arange(0, n_bins + 1) / n_bins * 100

    if bucket_type == "bins":
        breakpoints = np.histogram(expected, bins=n_bins)[1]
    elif bucket_type == "quantiles":
        breakpoints = np.percentile(expected, breakpoints)

    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)

    # Избегаем деления на 0
    expected_percents = np.clip(expected_percents, 0.0001, None)
    actual_percents = np.clip(actual_percents, 0.0001, None)

    psi_value = np.sum((expected_percents - actual_percents) * np.log(expected_percents / actual_percents))
    return psi_value

def calculate_psi(expected: np.ndarray, actual: np.ndarray, bucket_type: str = "bins", n_bins: int = 10, axis: int = 0) -> np.ndarray:
    """Calculate PSI for 1D or 2D arrays."""
    if len(expected.shape) == 1:
        return _psi(expected, actual, bucket_type, n_bins)
    else:
        psi_values = []
        if axis == 0:
            for i in range(expected.shape[1]):
                psi_values.append(_psi(expected[:, i], actual[:, i], bucket_type, n_bins))
        elif axis == 1:
            for i in range(expected.shape[0]):
                psi_values.append(_psi(expected[i, :], actual[i, :], bucket_type, n_bins))
        return np.array(psi_values)

In [14]:
# Генерация датасетов

np.random.seed(42) # для одинаковой воспроизводимости при повторном запуске
SAMPLE_SIZE = 5000

# Базовый датасет с "чистым" нормальным распределением
data_base = np.random.normal(loc=50, scale=10, size=SAMPLE_SIZE)

# Второй датасет с нормальным распределением и небольшими изменениями (смещение среднего на +1 и увеличение разброса на 0.5 соответственно параметрам)
data_good = np.random.normal(loc=51, scale=10.5, size=SAMPLE_SIZE)

# Третий датасет с аномалиями
data_bad = np.concatenate([
    np.random.normal(loc=50, scale=10, size=int(SAMPLE_SIZE * 0.7)), # нормальные данные из базового датасета
    np.random.normal(loc=150, scale=5, size=int(SAMPLE_SIZE * 0.25)), # смещенное распределение
    np.random.uniform(low=0, high=10, size=int(SAMPLE_SIZE * 0.05)) # шум
])

In [15]:
# Создание DataFrame для анализа
df_base = pd.DataFrame(data_base, columns=['Значение'])
df_good = pd.DataFrame(data_good, columns=['Значение'])
df_bad = pd.DataFrame(data_bad, columns=['Значение'])

In [16]:
# Вывод статистик по каждому датасету
print("Статистика по чистому датасету:")
print(df_base.describe())
print("\nСтатистика по датасету с нормальными данными:")
print(df_good.describe())
print("\nСтатистика по датасету с аномалиями:")
print(df_bad.describe())

Статистика по чистому датасету:
        Значение
count 5,000.0000
mean     50.0560
std       9.9648
min      17.5873
25%      43.4209
50%      50.1347
75%      56.6601
max      89.2624

Статистика по датасету с нормальными данными:
        Значение
count 5,000.0000
mean     50.8963
std      10.6096
min       9.8148
25%      43.7909
50%      50.8168
75%      58.1110
max      88.0551

Статистика по датасету с аномалиями:
        Значение
count 5,000.0000
mean     72.7970
std      46.5143
min       0.0393
25%      44.3947
50%      53.5046
75%      96.5549
max     166.4286


In [17]:
# Расчет PSI между датасетами
psi_normal = _psi(data_base, data_good, bucket_type="bins")
psi_anomaly = _psi(data_base, data_bad, bucket_type="bins")

print(f"PSI между чистым датасетом и датасетом с нормальными данными: {psi_normal:.4f}")
print(f"PSI между чистым датасетом и датасетом с аномалиями: {psi_anomaly:.4f}")

PSI между чистым датасетом и датасетом с нормальными данными: 0.0156
PSI между чистым датасетом и датасетом с аномалиями: 0.1090


**Выводы:**
1. Среднее значение:
Чистый датасет и датасет с нормальными данными имеют близкие средние значения (≈ 50), а датасет с аномалиями имеет значительно большее среднее значение (72.7970).
* Стандартное отклонение:
У чистого датасета стандартное отклонение относительно низкое. Датасет с нормальными данными показывает чуть большую дисперсию (10.6096). Аномалии приводят к значительному увеличению стандартного отклонения (46.5143), что говорит о сильной вариабельности значений.
* Минимальные и максимальные значения:
Оба чистых датасета показывают примерно одинаковые диапазоны значений от ≈ 17 до ≈ 89. Появление аномалий резко увеличивает диапазон — минимальное значение падает почти до нуля, максимальное же возрастает до 166.4286, что является значительным выбросом.

Нормализация данных практически не влияет на распределение (значение PSI = 0.0156).

Присутствие аномалий сильно искажает статистику и вызывает значительный популяционный сдвиг (PSI = 0.1090).